## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [1]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [3]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [4]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [5]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [6]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [7]:
# Write collection_bbox(features) and apply to all four LOD files
# Your code here
def collection_bbox(features):
    boxes = [feature_bbox(f) for f in features]
    lon_min = min(b[0] for b in boxes)
    lat_min = min(b[1] for b in boxes)
    lon_max = max(b[2] for b in boxes)
    lat_max = max(b[3] for b in boxes)
    return [lon_min, lat_min, lon_max, lat_max]


lod_dir = Path("../../data/lod")
lod_files = {
    "coarse": "railroads_coarse.geojson",
    "medium": "railroads_medium.geojson",
    "fine": "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'Level':<12} {'lon_min':>10} {'lat_min':>10} {'lon_max':>10} {'lat_max':>10}")
print("-" * 56)

for name, fname in lod_files.items():
    with open(lod_dir / fname) as f:
        fc = json.load(f)
    bbox = collection_bbox(fc["features"])
    print(f"{name:<12} {bbox[0]:>10.2f} {bbox[1]:>10.2f} {bbox[2]:>10.2f} {bbox[3]:>10.2f}")

Level           lon_min    lat_min    lon_max    lat_max
--------------------------------------------------------
coarse          -123.01     -41.48     150.96      60.98
medium          -150.11     -51.89     179.36      69.60
fine            -150.11     -51.89     179.36      69.60
extra_fine      -150.11     -51.90     179.36      69.60


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [8]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson
# Your code here
with open(lod_dir / "railroads_fine.geojson") as f:
    fine = json.load(f)

areas = []
for feat in fine["features"]:
    bbox = feature_bbox(feat)
    area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
    areas.append((area, bbox, feat["properties"].get("category", "?")))

areas.sort(reverse=True)

print(f"{'Area':>10}  {'Category':<20}  Bbox")
print("-" * 70)
for area, bbox, cat in areas[:5]:
    print(f"{area:>10.2f}  {cat:<20}  {bbox}")

      Area  Category              Bbox
----------------------------------------------------------------------
     29.31  0                     [90.609351, 29.645107, 94.942815, 36.409271]
     18.65  0                     [132.255704, -23.549819, 134.323448, -14.530934]
     17.70  3                     [-64.094167, -26.192499, -58.156944, -23.210555]
     17.39  2                     [14.021914, 54.802728, 20.000001, 57.711109]
     12.35  2                     [-49.137778, -5.491666, -44.372222, -2.899167]


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

imagine two railroads that both run from the southwest corner of texas up to the northeast corner — one goes straight diagonal and the other zigzags all over but ends at the same point. their bboxes would be identical even tho the actual lines look totally different. for culling its fine tho, the bbox is just used to ask "could this feature possibly be in the viewport" and both features genuinely could be, so we still load and draw them correctly.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.